# Managing Prompts with seperate configuration files

In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG Pipeline Prompt

In [2]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [5]:
print(build_prompt("- Item1\n- Item2", "Which items?"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- Item1
- Item2

Question:
Which items?    



### Jinja Templates

In [10]:
def render_jinja_template(preprocessed_context, question):
    prompt = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
"""

    template = Template(prompt)
    rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)
    return rendered_template

In [11]:
print(render_jinja_template("- Item1\n- Item2", "Which items?"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- Item1
- Item2

Question:
Which items?    


### Load template from configuration file

In [ ]:
def template_from_config(yaml_path, prompt_key):
    with open(yaml_path, "r") as f:
        try:
            config = yaml.safe_load(f)
        except:
            print(f"Couldn't load config file at location: {yaml_path}")
            return 

    content = config["prompts"][prompt_key]
    template = Template(content)
    return template

In [ ]:
product_rag_prompt = template_from_config("./05-prompt/rag-prompts.yaml", "rag-prompt-products")

In [14]:
def render_template(template, **kwargs):
    rendered = template.render(**kwargs)
    return rendered

In [16]:
print(
    render_template(
    product_rag_prompt, 
    context="- Item 2\n- Item 5\n- Item 54", 
    question="Which headphones are available?"
    )
)

You are a helpful shopping assistant.

Use the following context to answer the question.

Context:
- Item 2
- Item 5
- Item 54

Question:
Which headphones are available?

Instructions:
- Answer based on the context only
- Do not answer outside of the context
- Do not use markdown formatting
- Do not use word "context" in the answer


### Using a prompt registry

In [17]:
ls_client = Client()

In [18]:
ls_template = ls_client.pull_prompt("product-rag")
ls_template

/Users/dom/Documents/GitHub/e2e-engineering-course/E2E-AI-Engineering-Bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'product-rag', 'lc_hub_commit_hash': 'c57a49236fe718126efad5ba6f256c68ef587dce883d243a1efb4de4686febd0'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful shopping assistant.\nUse the following context to answer the question.\nContext:\n{{ context }}\nQuestion:\n{{ question }}\nInstructions:\n- Answer based on the context only\n- Do not answer outside of the context\n- Do not use markdown formatting\n- Do not use word "context" in the answer\n'), additional_kwargs={})])

In [19]:
print(ls_template.messages[0].prompt.template)

You are a helpful shopping assistant.
Use the following context to answer the question.
Context:
{{ context }}
Question:
{{ question }}
Instructions:
- Answer based on the context only
- Do not answer outside of the context
- Do not use markdown formatting
- Do not use word "context" in the answer



In [23]:
def render_prompt_template_registry(prompt_name, **kwargs):
    template = ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    rendered = render_template(Template(template), **kwargs)
    return rendered

In [24]:
print(
    render_prompt_template_registry(
        "product-rag",
        context="- Item 2\n- Item 5\n- Item 54", 
        question="Which headphones are available?"
    )
)

You are a helpful shopping assistant.
Use the following context to answer the question.
Context:
- Item 2
- Item 5
- Item 54
Question:
Which headphones are available?
Instructions:
- Answer based on the context only
- Do not answer outside of the context
- Do not use markdown formatting
- Do not use word "context" in the answer
